# Notebook 1: Collecte des Données

## Projet: Modélisation Hydrologique pour la Prévision des Crues
## Bassin Versant de l'Agnéby, Côte d'Ivoire

Ce notebook collecte et prépare les données nécessaires pour la modélisation hydrologique.

## 1. Importation des bibliothèques

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import os

# Pour les données géospatiales
import rasterio
from rasterio.plot import show
import geopandas as gpd

# Configuration pour les graphiques
plt.style.use('seaborn')
sns.set_palette("husl")

print("Bibliothèques importées avec succès!")

## 2. Chargement des Données Hydrologiques

Les données hydrologiques proviennent des stations de mesure sur la rivière Agnéby.

In [ ]:
# Chemin vers les données hydrologiques
hydrological_path = '../data/hydrological/'

# Exemple de chargement de données de débit
# Remplacez ceci par vos vraies données
def load_hydrological_data(filepath):
    """
    Charge les données hydrologiques depuis un fichier CSV.
    
    Args:
        filepath: Chemin vers le fichier CSV
    
    Returns:
        DataFrame pandas avec les données
    """
    df = pd.read_csv(filepath)
    df['date'] = pd.to_datetime(df['date'])
    df.set_index('date', inplace=True)
    return df

# Exemple de structure de données (à remplacer avec vos vraies données)
sample_data = {
    'date': pd.date_range(start='2010-01-01', end='2024-12-31', freq='D'),
    'discharge_m3_s': np.random.uniform(10, 500, 5114),  # Débit en m³/s
    'water_level_m': np.random.uniform(1, 5, 5114),  # Niveau d'eau en m
    'precipitation_mm': np.random.exponential(5, 5114)  # Précipitation en mm
}

hydrological_df = pd.DataFrame(sample_data)
hydrological_df.set_index('date', inplace=True)

print(f"Données hydrologiques chargées: {len(hydrological_df)} jours")
print(hydrological_df.head())

## 3. Chargement des Données Satellitaires (depuis Earth Engine)

In [ ]:
# Chemin vers les données satellites exportées depuis Earth Engine
satellite_path = '../data/satellite/'

def load_geotiff(filepath):
    """
    Charge un fichier GeoTIFF.
    
    Args:
        filepath: Chemin vers le fichier GeoTIFF
    
    Returns:
        Données raster et métadonnées
    """
    with rasterio.open(filepath) as src:
        data = src.read(1)
        meta = src.meta
    return data, meta

# Exemple: Charger le MNT
# mnt_data, mnt_meta = load_geotiff(satellite_path + 'agneby_elevation.tif')

print("Fonctions de chargement satellite définies")

## 4. Chargement des Données Climatiques

In [ ]:
# Chargement des données climatiques (CHIRPS, ERA5)
climate_path = '../data/climate/'

def load_climate_data(filepath):
    """
    Charge les données climatiques.
    
    Args:
        filepath: Chemin vers le fichier
    
    Returns:
        DataFrame pandas
    """
    df = pd.read_csv(filepath)
    df['date'] = pd.to_datetime(df['date'])
    df.set_index('date', inplace=True)
    return df

# Exemple de données climatiques
climate_data = {
    'date': pd.date_range(start='2000-01-01', end='2024-12-31', freq='M'),
    'temperature_C': 25 + 5 * np.sin(np.linspace(0, 4*np.pi, 300)) + np.random.normal(0, 1, 300),
    'precipitation_mm': np.random.exponential(100, 300)
}

climate_df = pd.DataFrame(climate_data)
climate_df.set_index('date', inplace=True)

print(f"Données climatiques chargées: {len(climate_df)} mois")
print(climate_df.head())

## 5. Chargement du MNT (Modèle Numérique de Terrain)

In [ ]:
# Chargement du MNT depuis SRTM
topography_path = '../data/topography/'

# Le MNT sera exporté depuis Earth Engine
# mnt_data, mnt_meta = load_geotiff(topography_path + 'agneby_elevation.tif')

print("MNT sera chargé depuis Earth Engine export")

## 6. Visualisation des Données

In [ ]:
# Visualisation des données hydrologiques
fig, axes = plt.subplots(3, 1, figsize=(15, 10))

# Débit
axes[0].plot(hydrological_df.index, hydrological_df['discharge_m3_s'])
axes[0].set_title('Débit de la rivière Agnéby (m³/s)')
axes[0].set_ylabel('Débit (m³/s)')
axes[0].grid(True)

# Niveau d'eau
axes[1].plot(hydrological_df.index, hydrological_df['water_level_m'])
axes[1].set_title('Niveau d\'eau (m)')
axes[1].set_ylabel('Niveau (m)')
axes[1].grid(True)

# Précipitation
axes[2].bar(hydrological_df.index, hydrological_df['precipitation_mm'], width=1)
axes[2].set_title('Précipitations journalières (mm)')
axes[2].set_ylabel('Précipitation (mm)')
axes[2].grid(True)

plt.tight_layout()
plt.savefig('../results/01_hydrological_data.png', dpi=300)
plt.show()

print("Graphique sauvegardé dans results/")

## 7. Statistiques Descriptives

In [ ]:
# Statistiques des données hydrologiques
print("=== STATISTIQUES DES DONNÉES HYDROLOGIQUES ===")
print(hydrological_df.describe())

# Statistiques des données climatiques
print("\n=== STATISTIQUES DES DONNÉES CLIMATIQUES ===")
print(climate_df.describe())

## 8. Sauvegarde des Données Préparées

In [ ]:
# Créer le dossier results s'il n'existe pas
os.makedirs('../results', exist_ok=True)

# Sauvegarder les données hydrologiques
hydrological_df.to_csv('../results/hydrological_data_prepared.csv')

# Sauvegarder les données climatiques
climate_df.to_csv('../results/climate_data_prepared.csv')

print("Données sauvegardées dans results/")

## 9. Prochaines Étapes

Après ce notebook:
1. Exécuter `02_preprocessing.ipynb` pour le prétraitement
2. Exécuter `03_eda.ipynb` pour l'analyse exploratoire
3. Exécuter `04_climate_analysis.ipynb` pour l'analyse climatique
4. Exécuter `05_model_training.ipynb` pour l'entraînement des modèles